## Abstract

Despite the header being `## Abstract`, this section will render as a highlighted section titled *Summary*. Ensure this section is a **maximum** of 280 characters.

----

:::{.callout-note title="AI usage disclosure" collapse="true"}
This is a placeholder for the AI usage disclosure. Once all authors sign the AI code form on Airtable, SlackBot will message you an AI disclosure that you should place here.
:::

## Purpose

Once edited by you, this file will become your publication. Alternatively, if you already have a notebook written that you're trying to transform into a pub, replace this file with your own, but be sure to add the YAML front matter (the first cell) to your notebook.

Your pub should begin with a section titled **Purpose** where you, as briefly as possible, explain why you did the work described in the pub, the key takeaway, your primary audience, and how you think it could be useful to them/why you're sharing it.

## Learn more

Refer to [Demo](/examples/demo.html) to learn about possible syntax.

## Demo: single-cell segmentation at scale

The cells below run the full pipeline end to end on a handful of example images: brightfield ND2s are segmented with Cellpose-SAM (via [`arcadia-microscopy-tools`](https://github.com/Arcadia-Science/arcadia-microscopy-tools)) into label masks, QC overlays, and per-cell measurement tables. The demo runs the same segmentation code in one of two modes, chosen automatically so the notebook is runnable whether or not you have our cloud infrastructure.

**Scaled mode (Modal + S3).** When the [Modal](https://modal.com/) client is authenticated and the `aws-s3-access-secret` (AWS credentials with read/write access to the `cloud-scaled-cell-phenotyping` bucket) is available, the heavy GPU compute is fanned out across Modal containers: each worker reads one ND2 from S3, segments it on a GPU, and writes the derived artifacts back to S3. The notebook only triggers the run and displays small previews returned by the workers. This is the path internal readers use, and it is what lets the pipeline scale to large datasets. One-time setup: `uv run modal setup`, plus creating the `aws-s3-access-secret` Modal secret.

**Local mode (no cloud account required).** When that infrastructure is not available, the notebook falls back to running the identical Cellpose-SAM pipeline on your own machine (CPU by default). It downloads a few example ND2s from the public [Zenodo](https://zenodo.org/) archive over HTTPS, segments them locally, and writes the artifacts to a local folder. This path needs no Modal account and no S3 credentials — only the local segmentation dependencies, installed with `uv sync --extra demo`. Because it runs on CPU, it is intentionally scoped to a few images via `limit`.

Either way, `run_demo` returns the same result object, so the display and analysis cells below are unchanged across modes. Set the environment variable `CSCP_FORCE_LOCAL=1` to force the local path, or pass `backend="modal"` / `backend="local"` explicitly.

In [ ]:
from analysis.runner import run_demo

# The example dataset: single-channel brightfield ND2s under `datasets/<DATASET_ID>/raw/`.
DATASET_ID = "20260731_140641_821"

# Segment the first few images. `run_demo` uses the scaled Modal + S3 path when that
# infrastructure is available, and otherwise falls back to running Cellpose-SAM
# locally on example images downloaded from the public Zenodo archive.
run = run_demo(DATASET_ID, limit=3)

if run.backend == "modal":
    print(f"[modal] Wrote artifacts to s3://cloud-scaled-cell-phenotyping/{run.output_prefix}/")
else:
    print(f"[local] Wrote artifacts to {run.output_prefix}")
print(f"Segmented {len(run.results)} image(s), {len(run.failures)} failure(s).")

### Segmentation overlays

The pipeline produces a small outline-overlay preview for each image (cell boundaries drawn on the brightfield image), which we display here as a quick visual QC of the segmentation.

In [ ]:
from pathlib import Path

from IPython.display import Image as IPyImage
from IPython.display import display

for summary in run.results:
    overlay_path = summary.get("local_overlay")
    if overlay_path:
        print(f"{Path(summary['input']).name}: {summary['n_cells']} cells")
        display(IPyImage(filename=overlay_path, width=500))

### Per-cell morphology

The pipeline also produces per-cell measurements (regionprops, converted to microns where the pixel size is known). We pool them across the demo images and plot a couple of shape features to show the kind of phenotyping this pipeline enables.

In [ ]:
import matplotlib.pyplot as plt

measurements = run.measurements
measurements.head()

In [ ]:
# Cell area (in microns^2 when the pixel size is available, else pixels) vs. eccentricity.
area_col = "area_um2" if "area_um2" in measurements.columns else "area"

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(measurements[area_col], measurements["eccentricity"], s=8, alpha=0.5)
ax.set_xlabel("cell area (um^2)" if area_col == "area_um2" else "cell area (px)")
ax.set_ylabel("eccentricity")
ax.set_title(f"Per-cell morphology (n = {len(measurements)} cells)")
plt.show()